In [1]:
!pip install tensorflow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.83.1-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.6.0-cp313-cp313-win_amd64.whl.metadata (8.8 kB)
  Using cach

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.0 which is incompatible.


In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from joblib import dump

# File paths (adapt if your files are in another folder)
DATA_PATH = r"C:\Users\ateek\Downloads\Twitter_Data.xlsx" # raw dataset
MODEL_PATH = r"C:\Users\ateek\Downloads\sentiment_model.h5"    # trained Keras model (for Streamlit)
TOKENIZER_PATH = r"C:\Users\ateek\Downloads\tokenizer.joblib"  # saved tokenizer for streamlit

# Reproducibility
RANDOM_STATE = 42

# Model & text hyperparameters (Used to build the architecture initially, and format text now)
MAX_NUM_WORDS = 20000        # vocabulary size for Tokenizer
MAX_SEQUENCE_LENGTH = 40     # max tokens per tweet (for padding)
EMBEDDING_DIM = 100          # dimension of embedding vectors
EPOCHS = 15                  # max epochs (Only used if initially building the model)
BATCH_SIZE = 32              # batch size

# Mapping between sentiment strings and numeric IDs
LABEL_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}
NUM_CLASSES = len(LABEL_TO_ID)
 
print("Python version:", tf.sysconfig.get_build_info().get("python_version", "N/A"))
print("TensorFlow version:", tf.__version__)

Python version: N/A
TensorFlow version: 2.21.0


In [3]:
df_raw = pd.read_excel(r"C:\Users\ateek\Downloads\Twitter_Data.xlsx")
print ("Raw Shape:", df_raw.shape)
print ("\nColumns:", df_raw.columns.tolist())

# Show first few rows
display(df_raw.head())

# Sentiment label distribution
print("\nSentiment value counts:")
print(df_raw["sentiment"].value_counts(dropna=False))

# Simple length distribution of raw text
df_raw["text_len"] = df_raw["text"].astype(str).str.len()
print("\nText length (characters) summary:")
print(df_raw["text_len"].describe())

Raw Shape: (27481, 4)

Columns: ['textID', 'text', 'selected_text', 'sentiment']


,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative



Sentiment value counts:
sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64

Text length (characters) summary:
count    27481.000000
mean        68.352571
std         35.625950
min          3.000000
25%         39.000000
50%         64.000000
75%         97.000000
max        159.000000
Name: text_len, dtype: float64


In [4]:
def clean_text(text: str) -> str:
    """
    Simple tweet cleaning.

    Steps:
    - Ensure input is a string
    - Remove URLs
    - Remove @mentions
    - Keep letters, spaces, ! and ?
    - Lowercase
    - Collapse multiple spaces
    """
    text = str(text)
    text = re.sub(r"http\S+", " ", text)        # remove URLs
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text) # remove @mentions

    # Keep letters, spaces, and basic sentiment punctuation ! and ?
    text = re.sub(r"[^a-zA-Z\s!?]", " ", text)
    
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Drop rows with missing text or sentiment
df = df_raw.dropna(subset=["text", "sentiment"]).copy()

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

# Drop very short cleaned tweets (1-2 characters)
df = df[df["clean_text"].str.len() > 2]

# Map labels to IDs
df["label_id"] = df["sentiment"].map(LABEL_TO_ID)

# Drop any rows with unmapped labels (should be none)
df = df.dropna(subset=["label_id"])
df["label_id"] = df["label_id"].astype(int)

print("After cleaning, shape:", df.shape)
display(df[["text", "clean_text", "sentiment", "label_id"]].head())

After cleaning, shape: (27470, 7)


,text,clean_text,sentiment,label_id
0,"I`d have responded, if I were going",i d have responded if i were going,neutral,1
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad i will miss you here in san diego!!!,negative,0
2,my boss is bullying me...,my boss is bullying me,negative,0
3,what interview! leave me alone,what interview! leave me alone,negative,0
4,"Sons of ****, why couldn`t they put them on t...",sons of why couldn t they put them on the rele...,negative,0


In [5]:
X = df["clean_text"].values    # numpy array of strings
y = df["label_id"].values      # numpy array of ints

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

print("\nTrain label distribution:")
print(pd.Series(y_train).map(ID_TO_LABEL).value_counts())

print("\nTest label distribution:")
print(pd.Series(y_test).map(ID_TO_LABEL).value_counts())

Train size: 21976, Test size: 5494

Train label distribution:
neutral     8888
positive    6866
negative    6222
Name: count, dtype: int64

Test label distribution:
neutral     2222
positive    1716
negative    1556
Name: count, dtype: int64


In [8]:
tokenizer = Tokenizer(num_words=MAX_NUM_WORDS, oov_token="<00V>")
tokenizer.fit_on_texts (X_train)

X_train_seq = tokenizer.texts_to_sequences (X_train)
X_test_seq = tokenizer.texts_to_sequences (X_test)

X_train_pad = pad_sequences (
    X_train_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences (
    X_test_seq,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("X_train_pad shape:", X_train_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

X_train_pad shape: (21976, 40)
X_test_pad shape: (5494, 40)


In [10]:
import warnings
warnings.filterwarnings("ignore")
from tensorflow.keras.layers import Bidirectional, LSTM

model = models.Sequential([
    layers.Embedding(
        input_dim=MAX_NUM_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    ),
    layers.SpatialDropout1D(0.1),
    Bidirectional(LSTM(128)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
import warnings
warnings.filterwarnings("ignore")
# Ensure numeric arrays
X_train_pad = np.asarray(X_train_pad, dtype="int32")
X_test_pad  = np.asarray(X_test_pad, dtype="int32")
y_train     = np.asarray(y_train, dtype="int32")
y_test      = np.asarray(y_test, dtype="int32")

# Compute class weights to handle any imbalance / ignored classes
classes = np.array([0, 1, 2]) # negative, neutral, positive
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_array)}
print("Class weights:", class_weight_dict)

# EarlyStopping callback
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Train model
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
    class_weight=class_weight_dict
)

Class weights: {0: 1.1773277617057751, 1: 0.8241824182418241, 2: 1.066899698999903}
Epoch 1/15
687/687 ━━━━━━━━━━━━━━━━━━━━ 190s 277ms/step - accuracy: 0.7763 - loss: 0.5409 - val_accuracy: 0.6988 - val_loss: 0.7179
Epoch 2/15
687/687 ━━━━━━━━━━━━━━━━━━━━ 235s 324ms/step - accuracy: 0.8412 - loss: 0.3987 - val_accuracy: 0.6822 - val_loss: 0.8328
Epoch 3/15
687/687 ━━━━━━━━━━━━━━━━━━━━ 267s 331ms/step - accuracy: 0.8850 - loss: 0.3032 - val_accuracy: 0.6873 - val_loss: 0.9234


In [18]:
# ------------------------------------------------------------------
# Evaluate on test set
# ------------------------------------------------------------------

y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

target_names = [ID_TO_LABEL[i] for i in range(NUM_CLASSES)]

print("Classification report:\n")
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

print("Confusion matrix (rows = true, cols = pred):")
print(confusion_matrix(y_test, y_pred))

172/172 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step
Classification report:

              precision    recall  f1-score   support

    negative       0.70      0.66      0.68      1556
     neutral       0.68      0.64      0.66      2222
    positive       0.72      0.81      0.76      1716

    accuracy                           0.70      5494
   macro avg       0.70      0.70      0.70      5494
weighted avg       0.70      0.70      0.70      5494

Confusion matrix (rows = true, cols = pred):
[[1033  399  124]
 [ 389 1416  417]
 [  61  265 1390]]


In [21]:
# ------------------------------------------------------------------
# Save model & tokenizer
# ------------------------------------------------------------------
model.save(MODEL_PATH)
dump(tokenizer, TOKENIZER_PATH)

print("Saved model to:", MODEL_PATH)
print("Saved tokenizer to:", TOKENIZER_PATH)

Saved model to: C:\Users\ateek\Downloads\sentiment_model.h5
Saved tokenizer to: C:\Users\ateek\Downloads\tokenizer.joblib
